# EBT Film Analysis for AIC144 Dataset

This notebook processes TIFF files from EBT film scans and converts them to dose distributions.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
import tifffile
import pandas as pd
from scipy import ndimage
from scipy.ndimage import gaussian_filter

## Configuration and Calibration

In [ ]:
@dataclass
class Calibration:
    """EBT3 calibration: dose = a0 + a1*netOD + a2*netOD^2 + a3*netOD^3"""
    a0: float = 0
    a1: float = 0
    a2: float = 0
    a3: float = 0

    def __call__(self, x):
        return self.a0 + self.a1*x + self.a2*x**2 + self.a3*x**3

    def __repr__(self):
        terms = []
        if self.a0 != 0:
            terms.append(f'{self.a0}')
        if self.a1 != 0:
            terms.append(f'{self.a1}*x')
        if self.a2 != 0:
            terms.append(f'{self.a2}*x^2')
        if self.a3 != 0:
            terms.append(f'{self.a3}*x^3')
        return 'f(x) = ' + ' + '.join(terms) if terms else 'f(x) = 0'

# EBT3 proton calibration for 20 Gy range
ebt3_proton_calib_20Gy = Calibration(a1=9.62189, a3=78.75125)
print(ebt3_proton_calib_20Gy)

@dataclass
class FileData:
    """Container for file-specific data including metadata and dose information."""
    stem: str
    path: Path
    dpi: float
    px_to_mm: float
    raw_image: np.ndarray
    dose_full: np.ndarray
    shape: tuple
    dose_cropped: np.ndarray = None
    crop_bbox: tuple = None

In [ ]:
def get_tiff_dpi(filepath: Path) -> float:
    """Extract DPI from TIFF file metadata.
    
    Parameters:
    -----------
    filepath : Path
        Path to TIFF file
        
    Returns:
    --------
    float
        DPI value (defaults to 150 if not found)
    """
    try:
        with tifffile.TiffFile(filepath) as tif:
            page = tif.pages[0]
            x_res = page.tags.get('XResolution')
            if x_res:
                dpi = x_res.value[0] / x_res.value[1]
                return dpi
    except Exception as e:
        print(f"Warning: Could not read DPI from {filepath}: {e}")
    return 150.0  # Default fallback

def get_px_to_mm(dpi: float) -> float:
    """Convert DPI to pixel-to-mm conversion factor.
    
    Parameters:
    -----------
    dpi : float
        Dots per inch
        
    Returns:
    --------
    float
        Pixel to mm conversion factor (mm/pixel)
    """
    return 25.4 / dpi  # 1 inch = 25.4 mm

# Find first TIFF file to extract DPI
data_dir = Path(r"C:\Users\grzanka\OneDrive - ifj.edu.pl\Projects\MB_foils\Publication_2026\raw_data\ebt_aic144")
first_tiff = next(data_dir.rglob('*.tif'), None)

if first_tiff:
    DPI = get_tiff_dpi(first_tiff)
    print(f"DPI extracted from first TIFF: {DPI}")
else:
    DPI = 150.0  # Fallback
    print(f"No TIFF file found, using default DPI: {DPI}")

PX_TO_MM = get_px_to_mm(DPI)
print(f'Pixel to mm conversion (default from first file): {PX_TO_MM:.6f} mm/pixel')

In [ ]:
def net_optical_density(image: np.ndarray, channel_no: int = 0) -> np.ndarray:
    """Calculate net optical density from raw RGB image.
    
    Parameters:
    -----------
    image : np.ndarray
        RGB image array with shape (height, width, 3)
    channel_no : int
        Color channel (0=R, 1=G, 2=B)
    
    Returns:
    --------
    np.ndarray
        Net optical density
    """
    # Background values for unexposed film
    bg_r = 42804.451 
    bg_g = 44273.485
    bg_b = 27929.477 
    bg = [bg_r, bg_g, bg_b]
    
    return np.log10(bg[channel_no] / image[:, :, channel_no])

In [ ]:
def ebt3_dose_Gy(image: np.ndarray, 
                 calib: Calibration = ebt3_proton_calib_20Gy, 
                 channel_no: int = 0) -> np.ndarray:
    """Convert raw image to dose in Gy using calibration.
    
    Parameters:
    -----------
    image : np.ndarray
        RGB image array
    calib : Calibration
        Calibration object
    channel_no : int
        Color channel (0=R, 1=G, 2=B)
    
    Returns:
    --------
    np.ndarray
        Dose in Gy
    """
    netOD = net_optical_density(image, channel_no)
    return calib(netOD)

## Data Directory Setup

In [ ]:
# Data directory
data_dir = Path(r'C:\Users\grzanka\OneDrive - ifj.edu.pl\Projects\MB_foils\Publication_2026\raw_data\ebt_aic144')
print(f'Data directory exists: {data_dir.exists()}')
print(f'Data directory: {data_dir}')

In [ ]:
# Find all TIFF files (recursively in subdirectories)
excluded_relative = {
    'bezszyby/przed/film_20260122_BEZSZYBY095.tif',
    'szyba1/film_20260122_szyba1_096.tif',
}

tiff_files = sorted(list(data_dir.glob('**/*.tif')) + list(data_dir.glob('**/*.tiff')))
tiff_files = [f for f in tiff_files if f.relative_to(data_dir).as_posix() not in excluded_relative]

print(f'Found {len(tiff_files)} TIFF files (after exclusions):')
for f in tiff_files:
    print(f'  {f.relative_to(data_dir)}')

## Helper Functions for Plotting

In [ ]:
def plot_2d_dose(dose_array: np.ndarray, 
                 px_to_mm: float,
                 title: str = 'Dose Distribution',
                 vmin: float = 0,
                 vmax: float = None,
                 white_threshold_percent: float = 1.0,
                 figsize=(10, 8)):
    """Plot 2D dose distribution with proper axes in mm.
    
    Parameters:
    -----------
    dose_array : np.ndarray
        2D dose array
    px_to_mm : float
        Pixel to mm conversion factor
    title : str
        Plot title
    vmin, vmax : float
        Color scale limits
    white_threshold_percent : float
        Percentage of vmax below which colors appear white
    figsize : tuple
        Figure size
    """
    from matplotlib.colors import LinearSegmentedColormap
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Create extent for proper axis scaling
    height, width = dose_array.shape
    extent = [0, width * px_to_mm, height * px_to_mm, 0]
    
    # Determine vmax if not provided
    if vmax is None:
        vmax = dose_array.max()
    
    # Create custom colormap: white -> green (50%) -> red (100%)
    threshold_frac = white_threshold_percent / 100.0
    cmap_dict = {
        'red':   [(0.0, 1.0, 1.0),
                  (threshold_frac, 1.0, 1.0),
                  (0.5, 0.0, 0.0),
                  (1.0, 1.0, 1.0)],
        'green': [(0.0, 1.0, 1.0),
                  (threshold_frac, 1.0, 1.0),
                  (0.5, 0.5, 0.5),
                  (1.0, 0.0, 0.0)],
        'blue':  [(0.0, 1.0, 1.0),
                  (threshold_frac, 1.0, 1.0),
                  (0.5, 0.0, 0.0),
                  (1.0, 0.0, 0.0)]
    }
    cmap = LinearSegmentedColormap('custom_dose', cmap_dict)
    
    im = ax.imshow(dose_array, cmap=cmap, vmin=vmin, vmax=vmax, 
                   extent=extent, aspect='equal')
    
    ax.set_xlabel('X [mm]', fontsize=12)
    ax.set_ylabel('Y [mm]', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    cbar = fig.colorbar(im, ax=ax, label='Dose [Gy]', fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    return fig, ax, im

In [ ]:
def plot_profiles(dose_array: np.ndarray,
                  px_to_mm: float,
                  x_position: int = None,
                  y_position: int = None,
                  profile_width: int = 5,
                  title: str = 'Dose Profiles',
                  figsize=(12, 5)):
    """Plot horizontal and vertical dose profiles.
    
    Parameters:
    -----------
    dose_array : np.ndarray
        2D dose array
    px_to_mm : float
        Pixel to mm conversion
    x_position : int
        Row index for horizontal profile (default: center)
    y_position : int
        Column index for vertical profile (default: center)
    profile_width : int
        Width of averaging region
    title : str
        Plot title
    figsize : tuple
        Figure size
    """
    height, width = dose_array.shape
    
    # Default positions at center
    if x_position is None:
        x_position = height // 2
    if y_position is None:
        y_position = width // 2
    
    # Calculate profiles with averaging
    hw = profile_width // 2
    
    # Horizontal profile (along x-axis at y=x_position)
    h_profile = dose_array[max(0, x_position-hw):min(height, x_position+hw), :].mean(axis=0)
    h_x_mm = np.arange(len(h_profile)) * px_to_mm
    
    # Vertical profile (along y-axis at x=y_position)
    v_profile = dose_array[:, max(0, y_position-hw):min(width, y_position+hw)].mean(axis=1)
    v_y_mm = np.arange(len(v_profile)) * px_to_mm
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    ax1.plot(h_x_mm, h_profile, 'b-', linewidth=2)
    ax1.set_xlabel('X Position [mm]', fontsize=11)
    ax1.set_ylabel('Dose [Gy]', fontsize=11)
    ax1.set_title(f'Horizontal Profile at Y={x_position*px_to_mm:.1f} mm', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, None)
    
    ax2.plot(v_y_mm, v_profile, 'r-', linewidth=2)
    ax2.set_xlabel('Y Position [mm]', fontsize=11)
    ax2.set_ylabel('Dose [Gy]', fontsize=11)
    ax2.set_title(f'Vertical Profile at X={y_position*px_to_mm:.1f} mm', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, None)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    return fig, (ax1, ax2), (h_profile, v_profile)

In [ ]:
def auto_crop_dose(dose_array: np.ndarray,
                   px_to_mm: float,
                   dose_threshold_gy: float = 0.5,
                   smooth_sigma_mm: float = 1.0,
                   min_diameter_mm: float = 10.0,
                   margin_mm: float = 5.0) -> tuple:
    """Crop dose array to the dominant irradiated region.

    Steps:
    1) Smooth dose to suppress noise.
    2) Threshold at dose_threshold_gy to find irradiated pixels.
    3) Keep the largest connected component that is big enough to hold a 1 cm diameter circle.
    4) Expand bounding box by margin_mm on each side.

    Returns (cropped_array, (y_min, y_max, x_min, x_max)).
    """
    # Ensure non-negative dose for masking
    dose_clipped = np.clip(dose_array, 0, None)

    sigma_px = max(0.5, smooth_sigma_mm / px_to_mm)
    margin_px = int(np.ceil(margin_mm / px_to_mm))
    min_radius_px = (min_diameter_mm / px_to_mm) / 2.0
    min_area_px = np.pi * (min_radius_px ** 2)

    # Smooth then threshold
    smoothed = gaussian_filter(dose_clipped, sigma=sigma_px)
    mask = smoothed > dose_threshold_gy

    if not mask.any():
        return dose_array, (0, dose_array.shape[0], 0, dose_array.shape[1])

    # Connected components
    labels, num = ndimage.label(mask)
    if num == 0:
        return dose_array, (0, dose_array.shape[0], 0, dose_array.shape[1])

    sizes = ndimage.sum(mask, labels, index=range(1, num + 1))
    valid_labels = [i + 1 for i, s in enumerate(sizes) if s >= min_area_px]

    # Pick largest valid component; if none valid, pick absolute largest
    if valid_labels:
        target_label = max(valid_labels, key=lambda lbl: sizes[lbl - 1])
    else:
        target_label = int(np.argmax(sizes)) + 1

    y_idx, x_idx = np.where(labels == target_label)
    if len(y_idx) == 0:
        return dose_array, (0, dose_array.shape[0], 0, dose_array.shape[1])

    y_min = max(0, y_idx.min() - margin_px)
    y_max = min(dose_array.shape[0], y_idx.max() + margin_px)
    x_min = max(0, x_idx.min() - margin_px)
    x_max = min(dose_array.shape[1], x_idx.max() + margin_px)

    cropped = dose_array[y_min:y_max, x_min:x_max]
    return cropped, (y_min, y_max, x_min, x_max)

## Process All TIFF Files

In [ ]:
# Cropping parameters
CROP_DOSE_THRESHOLD_GY = 0.5
CROP_SMOOTH_SIGMA_MM = 0.1
CROP_MIN_DIAMETER_MM = 5.0
CROP_MARGIN_MM = 1.0

# Colormap parameters
CMAP_WHITE_THRESHOLD_PERCENT = 1.0  # White for doses below this % of max

In [ ]:
# Process each TIFF file
processed_data = {}

for tiff_file in tiff_files:
    print(f'\n{"="*60}')
    print(f'Processing: {tiff_file.name}')
    print(f'{"="*60}')
    
    # Read image
    im = tifffile.imread(tiff_file)
    print(f'Image shape: {im.shape}, dtype: {im.dtype}')
    
    # Extract DPI from this specific file
    file_dpi = get_tiff_dpi(tiff_file)
    file_px_to_mm = get_px_to_mm(file_dpi)
    print(f'DPI: {file_dpi:.1f}, px_to_mm: {file_px_to_mm:.6f} mm/pixel')
    
    # Convert to dose
    dose_Gy = ebt3_dose_Gy(im)
    print(f'Dose range: [{dose_Gy.min():.2f}, {dose_Gy.max():.2f}] Gy')
    
    # Create FileData object
    file_data = FileData(
        stem=tiff_file.stem,
        path=tiff_file,
        dpi=file_dpi,
        px_to_mm=file_px_to_mm,
        raw_image=im,
        dose_full=dose_Gy,
        shape=im.shape
    )
    processed_data[tiff_file.stem] = file_data

print(f'\n{"="*60}')
print(f'Total files processed: {len(processed_data)}')
print(f'{"="*60}')

## Visualize Individual Files

For each file, we'll create:
1. Full dose distribution
2. Auto-cropped region with significant dose
3. Horizontal and vertical profiles

In [ ]:
# Select a file to visualize (change index or name as needed)
if processed_data:
    stem = list(processed_data.keys())[0]  # First file
    file_data = processed_data[stem]
    rel_path = file_data.path.relative_to(data_dir)
    print(f'Visualizing: {rel_path}')
else:
    print('No files processed yet')
    stem = None

In [ ]:
if stem:
    file_data = processed_data[stem]
    rel_path = file_data.path.relative_to(data_dir)
    
    # 1. Plot full dose distribution
    dose_full = file_data.dose_full
    vmax = np.percentile(dose_full[dose_full > 0], 99) if (dose_full > 0).any() else dose_full.max()
    
    fig, ax, im = plot_2d_dose(dose_full, 
                                px_to_mm=file_data.px_to_mm,
                                title=f'Full Dose Distribution - {rel_path}',
                                vmin=0, 
                                vmax=vmax,
                                white_threshold_percent=CMAP_WHITE_THRESHOLD_PERCENT,
                                figsize=(12, 10))
    plt.show()
    
    # 2. Auto-crop and plot
    dose_cropped, bbox = auto_crop_dose(
        dose_full,
        px_to_mm=file_data.px_to_mm,
        dose_threshold_gy=CROP_DOSE_THRESHOLD_GY,
        smooth_sigma_mm=CROP_SMOOTH_SIGMA_MM,
        min_diameter_mm=CROP_MIN_DIAMETER_MM,
        margin_mm=CROP_MARGIN_MM,
    )
    print(f'Cropped region: Y[{bbox[0]}:{bbox[1]}], X[{bbox[2]}:{bbox[3]}]')
    print(f'Cropped size before rotation: {dose_cropped.shape}')
    
    # Rotate 90 degrees anti-clockwise
    dose_cropped = np.rot90(dose_cropped, k=1)
    print(f'Cropped size after rotation: {dose_cropped.shape}')
    
    vmax_crop = np.percentile(dose_cropped[dose_cropped > 0], 99) if (dose_cropped > 0).any() else dose_cropped.max()
    
    # Plot cropped dose with profile lines
    height, width = dose_cropped.shape
    y_pos = height // 2
    x_pos = width // 2
    y_pos_mm = y_pos * file_data.px_to_mm
    x_pos_mm = x_pos * file_data.px_to_mm
    
    fig, ax, im = plot_2d_dose(dose_cropped,
                                px_to_mm=file_data.px_to_mm,
                                title=f'Cropped Dose Distribution - {rel_path}',
                                vmin=0,
                                white_threshold_percent=CMAP_WHITE_THRESHOLD_PERCENT,
                                figsize=(10, 8))
    # Add profile position lines
    ax.axhline(y=y_pos_mm, color='blue', linestyle='--', linewidth=1.5, alpha=0.8, label='Horizontal profile')
    ax.axvline(x=x_pos_mm, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label='Vertical profile')
    ax.legend(loc='upper right', fontsize=8)
    plt.show()
    
    # 3. Plot profiles
    fig, axes, profiles = plot_profiles(dose_cropped,
                                        px_to_mm=file_data.px_to_mm,
                                        title=f'Dose Profiles - {rel_path}',
                                        profile_width=10)
    plt.show()
    
    # Store cropped data (rotated)
    file_data.dose_cropped = dose_cropped
    file_data.crop_bbox = bbox

## Summary of All Files

In [ ]:
# Create a summary dataframe
summary_data = []

for stem, file_data in processed_data.items():
    dose = file_data.dose_full
    rel_path = file_data.path.relative_to(data_dir)
    summary_data.append({
        'File (relative)': rel_path,
        'DPI': f"{file_data.dpi:.1f}",
        'Width [px]': file_data.shape[1],
        'Height [px]': file_data.shape[0],
        'Width [mm]': f"{file_data.shape[1] * file_data.px_to_mm:.1f}",
        'Height [mm]': f"{file_data.shape[0] * file_data.px_to_mm:.1f}",
        'Min Dose [Gy]': f"{dose.min():.2f}",
        'Max Dose [Gy]': f"{dose.max():.2f}",
        'Mean Dose [Gy]': f"{dose.mean():.2f}",
        'Std Dose [Gy]': f"{dose.std():.2f}"
    })

df_summary = pd.DataFrame(summary_data)
print('\nSummary of all processed files:')
print(df_summary.to_string(index=False))

In [ ]:
# Loop through all files and create plots
for stem, file_data in processed_data.items():
    rel_path = file_data.path.relative_to(data_dir)
    
    print(f'\n{"="*60}')
    print(f'Visualizing: {rel_path}')
    print(f'{"="*60}\n')
    
    dose_full = file_data.dose_full
    
    # Auto-crop
    dose_cropped, bbox = auto_crop_dose(
        dose_full,
        px_to_mm=file_data.px_to_mm,
        dose_threshold_gy=CROP_DOSE_THRESHOLD_GY,
        smooth_sigma_mm=CROP_SMOOTH_SIGMA_MM,
        min_diameter_mm=CROP_MIN_DIAMETER_MM,
        margin_mm=CROP_MARGIN_MM,
    )
    
    # Rotate 90 degrees anti-clockwise
    dose_cropped = np.rot90(dose_cropped, k=1)
    
    vmax_crop = np.percentile(dose_cropped[dose_cropped > 0], 99) if (dose_cropped > 0).any() else dose_cropped.max()
    
    # Create custom colormap
    from matplotlib.colors import LinearSegmentedColormap
    threshold_frac = CMAP_WHITE_THRESHOLD_PERCENT / 100.0
    cmap_dict = {
        'red':   [(0.0, 1.0, 1.0),
                  (threshold_frac, 1.0, 1.0),
                  (0.5, 0.0, 0.0),
                  (1.0, 1.0, 1.0)],
        'green': [(0.0, 1.0, 1.0),
                  (threshold_frac, 1.0, 1.0),
                  (0.5, 0.5, 0.5),
                  (1.0, 0.0, 0.0)],
        'blue':  [(0.0, 1.0, 1.0),
                  (threshold_frac, 1.0, 1.0),
                  (0.5, 0.0, 0.0),
                  (1.0, 0.0, 0.0)]
    }
    cmap = LinearSegmentedColormap('custom_dose', cmap_dict)
    
    # Create figure with 3 subplots: 2D dose + 2 profiles
    fig = plt.figure(figsize=(16, 5))
    
    # 2D dose plot
    ax1 = plt.subplot(1, 3, 1)
    height, width = dose_cropped.shape
    extent = [0, width * file_data.px_to_mm, height * file_data.px_to_mm, 0]
    im = ax1.imshow(dose_cropped, cmap=cmap, vmin=0, vmax=vmax_crop, 
                    extent=extent, aspect='equal')
    ax1.set_xlabel('X [mm]')
    ax1.set_ylabel('Y [mm]')
    ax1.set_title(f'2D Dose Distribution - {rel_path}')
    plt.colorbar(im, ax=ax1, label='Dose [Gy]', fraction=0.046)
    
    # Calculate profile positions (in mm)
    y_pos = height // 2
    x_pos = width // 2
    y_pos_mm = y_pos * file_data.px_to_mm
    x_pos_mm = x_pos * file_data.px_to_mm
    
    # Add profile position lines on 2D dose plot
    ax1.axhline(y=y_pos_mm, color='blue', linestyle='--', linewidth=1.5, alpha=0.8, label='Horizontal profile')
    ax1.axvline(x=x_pos_mm, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label='Vertical profile')
    ax1.legend(loc='upper right', fontsize=8)
    
    # Horizontal profile
    ax2 = plt.subplot(1, 3, 2)
    h_profile = dose_cropped[max(0, y_pos-5):min(height, y_pos+5), :].mean(axis=0)
    h_x_mm = np.arange(len(h_profile)) * file_data.px_to_mm
    ax2.plot(h_x_mm, h_profile, 'b-', linewidth=2)
    ax2.set_xlabel('X Position [mm]')
    ax2.set_ylabel('Dose [Gy]')
    ax2.set_title(f'Horizontal Profile (Y={y_pos_mm:.1f} mm)')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, None)
    
    # Vertical profile
    ax3 = plt.subplot(1, 3, 3)
    v_profile = dose_cropped[:, max(0, x_pos-5):min(width, x_pos+5)].mean(axis=1)
    v_y_mm = np.arange(len(v_profile)) * file_data.px_to_mm
    ax3.plot(v_y_mm, v_profile, 'r-', linewidth=2)
    ax3.set_xlabel('Y Position [mm]')
    ax3.set_ylabel('Dose [Gy]')
    ax3.set_title(f'Vertical Profile (X={x_pos_mm:.1f} mm)')
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, None)
    
    fig.suptitle(rel_path, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Store cropped data (rotated)
    file_data.dose_cropped = dose_cropped
    file_data.crop_bbox = bbox

## Profile Comparison Across All Files

In [ ]:
# Compare profiles from all files
fig, (ax_h, ax_v) = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(processed_data)))

for idx, (stem, file_data) in enumerate(processed_data.items()):
    if file_data.dose_cropped is None:
        continue
    
    rel_path = file_data.path.relative_to(data_dir)
    dose_cropped = file_data.dose_cropped
    px_to_mm = file_data.px_to_mm
    height, width = dose_cropped.shape
    
    # Calculate profiles at center
    y_pos = height // 2
    x_pos = width // 2
    
    # Horizontal profile
    h_profile = dose_cropped[max(0, y_pos-5):min(height, y_pos+5), :].mean(axis=0)
    h_x_mm = np.arange(len(h_profile)) * px_to_mm
    ax_h.plot(h_x_mm, h_profile, '-', linewidth=2, color=colors[idx], 
              label=f'{rel_path.parent.name} (DPI: {file_data.dpi:.0f})')
    
    # Vertical profile
    v_profile = dose_cropped[:, max(0, x_pos-5):min(width, x_pos+5)].mean(axis=1)
    v_y_mm = np.arange(len(v_profile)) * px_to_mm
    ax_v.plot(v_y_mm, v_profile, '-', linewidth=2, color=colors[idx], 
              label=f'{rel_path.parent.name} (DPI: {file_data.dpi:.0f})')

# Format horizontal profile plot
ax_h.set_xlabel('X Position [mm]', fontsize=12)
ax_h.set_ylabel('Dose [Gy]', fontsize=12)
ax_h.set_title('Horizontal Profiles Comparison', fontsize=14, fontweight='bold')
ax_h.grid(True, alpha=0.3)
ax_h.set_ylim(0, None)
ax_h.legend(loc='best', fontsize=10)

# Format vertical profile plot
ax_v.set_xlabel('Y Position [mm]', fontsize=12)
ax_v.set_ylabel('Dose [Gy]', fontsize=12)
ax_v.set_title('Vertical Profiles Comparison', fontsize=14, fontweight='bold')
ax_v.grid(True, alpha=0.3)
ax_v.set_ylim(0, None)
ax_v.legend(loc='best', fontsize=10)

plt.tight_layout()
plt.show()